In [ ]:
#pentru a putea folosi TPU, am sters JAX peuntru a preveni conflictele de dependente
# si am instalat versiunea de TF 2.18.0 optimizata pt  si driverele libtpu de la Google
!export PATH="${HOME}/.local/bin:${PATH}" && uv pip uninstall --system jax jaxlib
!export PATH="${HOME}/.local/bin:${PATH}" && uv pip install --system tensorflow-tpu==2.18.0 --find-links https://storage.googleapis.com/libtpu-tf-releases/index.html

In [ ]:
import tensorflow as tf
from kaggle_datasets import KaggleDatasets
import numpy as np
import keras

print("Tensorflow version " + tf.__version__)
print("kaggle version" +  keras.__version__)

In [ ]:
#initializare TPU
import tensorflow as tf

try:
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver(tpu='local') 
    print('Running on TPU ', tpu.master())
    
    tf.config.experimental_connect_to_cluster(tpu)
    tf.tpu.experimental.initialize_tpu_system(tpu)
    strategy = tf.distribute.TPUStrategy(tpu)
    
except ValueError as e:
    print("TPU connection failed:", e)
    tpu = None
    strategy = tf.distribute.get_strategy()

print("REPLICAS: ", strategy.num_replicas_in_sync)

In [ ]:
# === Preprocesare și Augmentare Date ===
#
#   - Train:      80% din directorul train_dir  (~9.817 imagini)
#   - Validare:   20% din directorul train_dir  (~2.454 imagini)
#   - Test:       directorul test_dir           (~3.068 imagini)
#
# Augmentare aplicată pe setul de antrenare:
#   - Rescaling [0, 1]
#   - Rotație aleatorie ±10%
#   - Translație ±10% (H și W)
#   - Zoom aleatoriu ±10%
#   - Flip orizontal
#   - Contrast aleatoriu ±20%
#
# Validarea și testul primesc exclusiv rescaling, fără augmentare.
# Batch size: 512.
# Pipeline optimizat cu shuffle(5000) + prefetch(AUTOTUNE).
import tensorflow as tf
from tensorflow.keras import layers

train_dir = '/kaggle/input/datasets/shuvoalok/raf-db-dataset/DATASET/train'
test_dir = '/kaggle/input/datasets/shuvoalok/raf-db-dataset/DATASET/test'

img_size = 48
batch_size = 512  # better for TPU v3-8
num_classes = 7
epoci = 100        # more epochs

AUTOTUNE = tf.data.AUTOTUNE

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir, validation_split=0.20, subset="training", seed=123,
    image_size=(img_size, img_size), batch_size=batch_size,
    color_mode="grayscale", label_mode="categorical"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir, validation_split=0.20, subset="validation", seed=123,
    image_size=(img_size, img_size), batch_size=batch_size,
    color_mode="grayscale", label_mode="categorical"
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir, image_size=(img_size, img_size), batch_size=batch_size,
    color_mode="grayscale", label_mode="categorical", shuffle=False
)

data_augmentation = tf.keras.Sequential([
    layers.Rescaling(1./255),
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),       
    layers.RandomTranslation(height_factor=0.05, width_factor=0.05),  # ← era 0.1
    layers.RandomZoom(0.05),          
    layers.RandomContrast(0.1)         
])

rescaling_only = tf.keras.Sequential([
    layers.Rescaling(1./255)
])

train_ds = (train_ds
    .unbatch()
    .batch(batch_size, drop_remainder=True)
    .shuffle(5000, reshuffle_each_iteration=True)
    .map(lambda x, y: (data_augmentation(x, training=True), y),
         num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE))

val_ds = (val_ds
    .unbatch()
    .batch(batch_size, drop_remainder=True)
    .map(lambda x, y: (rescaling_only(x, training=False), y),
         num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE))

test_ds = (test_ds
    .unbatch()
    .batch(batch_size, drop_remainder=True)
    .map(lambda x, y: (rescaling_only(x, training=False), y),
         num_parallel_calls=AUTOTUNE)
    .prefetch(AUTOTUNE))

In [ ]:
# === Vizualizare Imagini Augmentate ===
#
# Extragem primul batch din train_ds și afișăm 9 imagini într-un grid 3×3.
# Imaginile sunt deja procesate de pipeline:
#   - Rescalate în intervalul [0, 1]
#   - Augmentate aleatoriu (rotație, zoom, flip, translație, contrast)
#
# Scopul: validare vizuală că pipeline-ul funcționează corect
# înainte de a începe antrenarea modelului.
import matplotlib.pyplot as plt
import numpy as np

print("Generăm variante augmentate corect...")

for images, labels in train_ds.take(1):
    plt.figure(figsize=(10, 10))
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().squeeze(), cmap='gray', vmin=0, vmax=1)
        plt.axis("off")
    plt.suptitle("Imagini din Train Dataset (Deja Augmentate și Scalate)", fontsize=16)
    plt.show()

In [ ]:
#pt raf-db aratam etichetele fiecarei emotii adica un numar 
clase = ['surprise', 'fear', 'disgust', 'happy', 'sad', 'angry', 'neutral']
class_labels = {name: i for i, name in enumerate(clase)}

print("Etichete (valabile pentru Antrenament, Validare și Test):")
print(class_labels)

print(f"\nVerificare clase detectate: {len(clase)} emoții")
for i, emotion in enumerate(clase):
    print(f"Index {i} (folder '{i+1}') -> {emotion}")

In [ ]:
# === Arhitectură V-CNN (Dogaru & Dogaru, 2023) ===
# Structură: 3 blocuri convoluționale cu [64, 128, 64] filtre, fiecare bloc conținând
# straturi Conv2D + ReLU, BatchNormalization, MaxPooling și Dropout(0.3).
# Capul de clasificare: GlobalAveragePooling2D → Dense(7, softmax).
# Compilat cu Adam + categorical crossentropy. Instanțiat în scopul strategiei TPU.
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv2D, Activation, BatchNormalization, 
                                     MaxPooling2D, Dropout, Flatten, 
                                     GlobalAveragePooling2D, Dense)

def create_v_cnn_model_rafdb(input_shape=(48, 48, 1), num_classes=7, 
                            flat=0, fil=[64, 128, 64], nl=[2, 1, 1], hid=[]):
    csize = 3
    stri = 2
    psiz = 4
    pad = 'same'
    drop1 = 0.5

    nfilmax = len(fil)
    model = Sequential()

    layer = 0
    if nl[layer] > 0:
        model.add(Conv2D(fil[layer], padding=pad, kernel_size=(csize, csize), input_shape=input_shape))
        model.add(Activation('relu'))
        for nonlin in range(1, nl[0]):
            model.add(Conv2D(fil[layer], padding=pad, kernel_size=(csize, csize)))
            model.add(Activation('relu'))
        model.add(Conv2D(fil[0], padding=pad, kernel_size=(csize, csize)))
        model.add(BatchNormalization())
        model.add(MaxPooling2D(pool_size=(psiz, psiz), strides=(stri, stri), padding=pad))
        model.add(Dropout(drop1))
    else:
        model.add(Conv2D(fil[0], padding=pad, kernel_size=(csize, csize), input_shape=input_shape))
        model.add(BatchNormalization())
        model.add(MaxPooling2D(pool_size=(psiz, psiz), strides=(stri, stri), padding=pad))
        model.add(Dropout(drop1))

    for layer in range(1, nfilmax):
        for nonlin in range(nl[layer]):
            model.add(Conv2D(fil[layer], padding=pad, kernel_size=(csize, csize)))
            model.add(Activation('relu'))
        model.add(Conv2D(fil[layer], padding=pad, kernel_size=(csize, csize)))
        model.add(BatchNormalization())
        model.add(MaxPooling2D(pool_size=(psiz, psiz), strides=(stri, stri), padding=pad))
        model.add(Dropout(drop1))

    if flat == 1:
        model.add(Flatten())
    else:
        model.add(GlobalAveragePooling2D())

    if len(hid) > 0:
        for units in hid:
            model.add(Dense(units, activation='relu'))
            model.add(Dropout(drop1))

    model.add(Dense(num_classes, activation='softmax'))

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

with strategy.scope():
    model = create_v_cnn_model_rafdb(
        input_shape=(48, 48, 1),
        num_classes=7,
        flat=0,
        fil=[64, 128, 64],
        nl=[2, 2, 1],
        hid=[]
    )

model.summary()

In [ ]:
batch_size = 128

In [ ]:
# Callbacks: salvare best model (val_accuracy), early stopping (patience=20),
# reducere LR la platou (factor=0.5, patience=7, min_lr=1e-6).
from keras.callbacks import ModelCheckpoint, EarlyStopping

checkpoint = ModelCheckpoint(
    "vcnn_best_model_rafdb.keras",
    monitor='val_accuracy', 
    verbose=1, 
    save_best_only=True, 
    mode='max'
)

early_stopping = EarlyStopping(
    monitor='val_accuracy', 
    patience=25,          
    restore_best_weights=True,
    verbose=1
)

lr_scheduler = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_accuracy', 
    factor=0.5, 
    patience=10,         
    min_lr=1e-5,          
    verbose=1
)

In [ ]:
import numpy as np

counts = np.array([1290, 281, 717, 4772, 1982, 705, 2524])
total = counts.sum()
n_classes = 7

class_weights = {i: total / (n_classes * counts[i]) for i in range(n_classes)}

print("Class weights:")
for i, w in class_weights.items():
    print(f"  Index {i}: {w:.4f}")

In [ ]:
# === Antrenare Model ===
# Se lansează antrenarea pe TPU pentru maximum 100 epoci, cu validare la fiecare epocă.
# Callbacks active: checkpoint (best model), early stopping (patience=20), ReduceLROnPlateau.
# Timpul de antrenare este cronometrat pentru a fi raportat ulterior.
import time

print("Începem antrenamentul pe TPU...")

t_start_train = time.time()
istoric_antrenare = model.fit(
    train_ds,
    epochs=epoci,
    validation_data=val_ds,
    callbacks=[checkpoint, early_stopping, lr_scheduler],
    class_weight=class_weights
)
t_end_train = time.time()

In [ ]:
# === Evaluare Model Final ===
# Se încarcă cel mai bun model salvat de checkpoint și se evaluează pe test set.
# Se raportează: timpul total de antrenare (minute), numărul de parametri,
# acuratețea pe test și latența medie de inferență per imagine (ms).
import time
import tensorflow as tf
timp_total_minute = int(t_end_train - t_start_train) / 60
print('Training with  ',epoci,' epochs, lasted  ',timp_total_minute,' minutes')

model_salvat = tf.keras.models.load_model("vcnn_best_model_rafdb.keras")

print('Număr total de parametri:', model_salvat.count_params())

t1_eval = time.time()
score = model_salvat.evaluate(test_ds, verbose=0)
t2_eval = time.time()

print(f'Cea mai buna acuratete de test: {score[1] * 100}%')

numar_imagini_test=3068

timp_total_ms = (t2_eval - t1_eval) * 1000
latenta_per_imagine = timp_total_ms / numar_imagini_test

print(f'Latența la predicție (per imagine): {latenta_per_imagine:.4f} ms')


In [ ]:
#graficele pentru evolutia acuratetii si scaderea erorii
#pentru setul de date de antrenament si validare
import matplotlib.pyplot as plt


acc = istoric_antrenare.history['accuracy']
val_acc = istoric_antrenare.history['val_accuracy']
loss = istoric_antrenare.history['loss']
val_loss = istoric_antrenare.history['val_loss']

epoci_rulate = range(1, len(acc) + 1)


plt.figure(figsize=(14, 5))


plt.subplot(1, 2, 1)
plt.plot(epoci_rulate, acc, 'b-', linewidth=2, label='Antrenament')
plt.plot(epoci_rulate, val_acc, 'r-', linewidth=2, label='Validare')
plt.title('Evoluția Acurateții V-CNN', fontsize=14)
plt.xlabel('Număr Epoci', fontsize=12)
plt.ylabel('Acuratețe', fontsize=12)
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.7)


plt.subplot(1, 2, 2)
plt.plot(epoci_rulate, loss, 'b-', linewidth=2, label='Eroare Antrenament')
plt.plot(epoci_rulate, val_loss, 'r-', linewidth=2, label='Eroare Validare')
plt.title('Scăderea Erorii (Loss) V-CNN', fontsize=14)
plt.xlabel('Număr Epoci', fontsize=12)
plt.ylabel('Valoare Eroare', fontsize=12)
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.7)


plt.tight_layout()
plt.show()

In [ ]:
import os
import matplotlib.pyplot as plt

clase_counts_train = {}
clase_counts_test = {}

for cls in sorted(os.listdir(train_dir)):
    train_path = os.path.join(train_dir, cls)
    test_path = os.path.join(test_dir, cls)
    clase_counts_train[cls] = len(os.listdir(train_path))
    clase_counts_test[cls] = len(os.listdir(test_path))

print("TRAIN:")
for cls, count in clase_counts_train.items():
    print(f"  Folder '{cls}': {count} imagini")

print("\nTEST:")
for cls, count in clase_counts_test.items():
    print(f"  Folder '{cls}': {count} imagini")

# Grafic
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(clase_counts_train.keys(), clase_counts_train.values(), color='steelblue')
axes[0].set_title('Distributie Train')
axes[0].set_xlabel('Clasa (folder)')

axes[1].bar(clase_counts_test.keys(), clase_counts_test.values(), color='salmon')
axes[1].set_title('Distributie Test')
axes[1].set_xlabel('Clasa (folder)')

plt.tight_layout()
plt.show()

In [ ]:
#afisarea matricei de confuzie
import numpy as np
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt

print("Calculăm predicțiile pe setul de test...")

y_pred_probs = model.predict(test_ds, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)

y_true = []
for images, labels in test_ds:
    y_true.extend(np.argmax(labels.numpy(), axis=1))
y_true = np.array(y_true)

etichete = clase 

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='PuRd', 
            xticklabels=etichete, yticklabels=etichete)
plt.title('Matricea de Confuzie - Model V-CNN Final')
plt.ylabel('Emoție Reală')
plt.xlabel('Emoție Prezisă')
plt.show()

print("\nRaport de Clasificare Detaliat:\n")
print(classification_report(y_true, y_pred, target_names=etichete, zero_division=0))

In [ ]:
# === Testare Vizuală pe Imagini Individuale ===
# Funcția testeaza_cu_legenda() primește o cale de imagine și modelul antrenat,
# preprocesează imaginea identic cu pipeline-ul de antrenare (48×48, grayscale, [0,1]),
# rulează predicția și afișează un grafic cu 2 panouri:
#   - Stânga: imaginea originală + emoția prezisă (cu procent de încredere)
#   - Dreapta: distribuția probabilităților softmax pentru toate 7 clase (în română)
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing import image

def testeaza_cu_legenda(cale_imagine, model):
    etichete_en = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
    
    clase_ro = {
        'angry': 'FURIE',
        'disgust': 'DEZGUST',
        'fear': 'FRICĂ',
        'happy': 'FERICIRE',
        'neutral': 'NEUTRU',
        'sad': 'TRISTEȚE',
        'surprise': 'SURPRIZĂ'
    }
    
    nume_ro_scurt = [clase_ro[e] for e in etichete_en]

    try:
        img_pt_model = image.load_img(cale_imagine, target_size=(48, 48), color_mode="grayscale")
    except Exception as e:
        print(f"Eroare la încărcarea imaginii: {e}")
        return

    img_array = image.img_to_array(img_pt_model)
    img_array = np.expand_dims(img_array, axis=0) 

    img_array = img_array / 255.0

    predictii_brute = model.predict(img_array, verbose=0)[0] 
    index_maxim = np.argmax(predictii_brute)
    
    emotie_w_en = etichete_en[index_maxim]
    emotie_w_ro = clase_ro[emotie_w_en]
    procent_w = predictii_brute[index_maxim] * 100

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    
    img_orig = cv2.imread(cale_imagine)
    if img_orig is None:
        print("Eroare: Nu am putut citi imaginea originală pentru afișare!")
        return
    img_orig = cv2.cvtColor(img_orig, cv2.COLOR_BGR2RGB)
    
    ax1.imshow(img_orig)
    ax1.axis('off') 
    
    culoare_titlu = 'green' if emotie_w_en == 'happy' else 'red' if emotie_w_en in ['angry', 'fear'] else 'darkblue'
    
    ax1.set_title(f"Rețeaua V-CNN prezice:\n{emotie_w_ro} ({procent_w:.2f}%)", 
                  fontsize=18, color=culoare_titlu, fontweight='bold', pad=20)

    y_pos = np.arange(len(nume_ro_scurt))
    procente_afisare = predictii_brute * 100
    
    bare = ax2.barh(y_pos, procente_afisare, align='center', color='skyblue', alpha=0.8)
    
    bare[index_maxim].set_color('deeppink')
    bare[index_maxim].set_alpha(1.0)

    ax2.set_yticks(y_pos)
    ax2.set_yticklabels(nume_ro_scurt, fontsize=12)
    ax2.invert_yaxis() 
    ax2.set_xlabel('Probabilitate (%)', fontsize=12)
    ax2.set_title('Distribuția detaliată a probabilităților', fontsize=14, pad=15)
    
    for i, bar in enumerate(bare):
        latime = bar.get_width()
        ax2.text(latime + 1, bar.get_y() + bar.get_height()/2, 
                 f'{procente_afisare[i]:.1f}%', 
                 va='center', fontsize=10, fontweight='bold' if i == index_maxim else 'normal')

    ax2.set_xlim(0, 110) 
    ax2.grid(axis='x', linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.show()

testeaza_cu_legenda('/kaggle/input/datasets/shuvoalok/raf-db-dataset/DATASET/test/4/test_0010_aligned.jpg', model)

In [ ]:
model.save('vcnn_best_model_fer_augmented.keras')

In [ ]:
#am creeat un dataset in care am salvat toate modelele antrenate
#aici aplicam modelul nostru schimband calea acestuia la nevoie
#trebuie rulata DUPA primele 3 celule 
from tensorflow.keras.models import load_model

cale_model = '/kaggle/input/datasets/mindrocrobert/modelulmeu/vcnn_model_1aprilie.keras'
model = load_model(cale_model)

print("Modelul este acum încărcat în memorie")